In [ ]:
"""
Huff-based preferred LPG distributor assignment per inhabited 1 km pixel (Nigeria)
using shortest-path travel time on friction-based raster graphs.

What this script does
---------------------
For each inhabited and traversable pixel, the script selects the preferred LPG
distributor for:
1) walking population
2) motorized population

The winner is selected with a Huff-style gravity score:

    score_ij = A_j / (T_ij + eps)^beta

where:
- A_j = distributor attractiveness
- T_ij = shortest-path travel time (minutes) from pixel i to distributor j
- beta = distance-decay exponent
- eps = small stabilization constant

Network and masking behavior
----------------------------
- Demand origins are only inhabited pixels from Population.tif (plus valid friction).
- Shortest paths are computed on a traversability graph built from valid walk/moto
  friction cells; paths may cross non-inhabited cells if friction is valid.
- Car mode is computed only when expected car users per pixel are >= 1; otherwise car results are copied from walk.
- No straight-line distance approximation is used.

Scalability and robustness
--------------------------
- Adaptive candidate preselection (60 km primary, then 100 km, then NN safety).
- One Dijkstra run per mode in the normal case.
- Component-aware fallback run used only when needed.
Outputs
-------
Main output: GPKG point layer (one point per processed inhabited pixel) with fields:
- car_share, walk_share
- best_reseller_id_walk, best_time_walk_min
- best_reseller_id_car,  best_time_car_min

Additional outputs:
- support rasters for winner IDs and travel times (walk and car)
- reseller lookup CSV
- optional profiling JSON log
"""

from __future__ import annotations

import json
import time
from typing import Iterable
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

import rasterio
from rasterio.warp import reproject, Resampling

# =====================================================================
# USER PARAMETERS
# =====================================================================
DATA_DIR = "dataset_big"

# Inputs
RESELLER_GPKG = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"
RESELLER_LAYER = "resell_and_filling"
POPULATION_RASTER = f"{DATA_DIR}/Population.tif"
CAR_SHARE_RASTER = f"{DATA_DIR}/vehicles_allocation_share.tif"
WALK_FRICTION_RASTER = f"{DATA_DIR}/friction_walk.tif"
MOTO_FRICTION_RASTER = f"{DATA_DIR}/friction_moto.tif"

# Outputs
OUTPUT_GPKG = f"{DATA_DIR}/huff_preferred_distributor_per_pixel.gpkg"
OUTPUT_GPKG_LAYER = "pixel_preference"

OUTPUT_RASTER_BEST_ID_WALK = f"{DATA_DIR}/huff_best_reseller_id_walk.tif"
OUTPUT_RASTER_BEST_TIME_WALK = f"{DATA_DIR}/huff_best_time_walk_min.tif"
OUTPUT_RASTER_BEST_ID_CAR = f"{DATA_DIR}/huff_best_reseller_id_car.tif"
OUTPUT_RASTER_BEST_TIME_CAR = f"{DATA_DIR}/huff_best_time_car_min.tif"
OUTPUT_LOOKUP_CSV = f"{DATA_DIR}/huff_reseller_lookup.csv"

# Preferred columns
RESELLER_ID_COLUMN = "id_res&fil"
ATTRACTIVENESS_COLUMN = "attractiveness"

# Huff parameters
BETA = 2.0
EPS = 1e-6
MIN_ATTRACTIVENESS = 1e-6

# Population rules
MIN_POP_PER_PIXEL = 0.0

# Car-share interpretation
CAR_SHARE_IS_PERCENT = True
MIN_EXPECTED_CAR_USERS_PER_PIXEL = 1.0

# Candidate strategy
MIN_CANDIDATES = 4
PRIMARY_SEARCH_RADIUS_KM = 60
MAX_SEARCH_RADIUS_KM = 100
EXTRA_CANDIDATES_CAR = 6

# Adaptive limits and safety fallback
INITIAL_LIMIT_FACTOR_WALK = 8.0
INITIAL_LIMIT_FACTOR_CAR = 10.0
FINAL_LIMIT_FACTOR_WALK = 14.0
FINAL_LIMIT_FACTOR_CAR = 16.0
LIMIT_MARGIN_MIN = 30.0
UNASSIGNED_TIME_MIN = 7000.0

# Graph assumptions
CELL_SIZE_METERS = 1000.0  # 1km grid
USE_8_NEIGHBORS = False

# Progress
PROGRESS_EVERY = 2000
MAX_PIXELS_DEBUG = None  # set int for quick testing

# Outputs flags
SAVE_SUPPORT_RASTERS = True

# Lightweight profiling
ENABLE_PROFILING = True
PROFILE_BLOCK_PIXELS = 10000
PROFILE_LOG_JSON = f"{DATA_DIR}/huff_run_profile.json"

# Nodata conventions
NODATA_FLOAT = -9999.0
NODATA_INT = -1

# =====================================================================
# HELPERS
# =====================================================================
def _read_raster(path: str):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile


def _align_to_reference(path: str, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    with rasterio.open(path) as src:
        dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _safe_attractiveness(values: Iterable) -> np.ndarray:
    arr = pd.to_numeric(pd.Series(values), errors="coerce").astype(np.float64).to_numpy()
    arr = np.where(np.isfinite(arr), arr, MIN_ATTRACTIVENESS)
    arr = np.maximum(arr, MIN_ATTRACTIVENESS)
    return arr


def _to_fraction(arr: np.ndarray) -> np.ndarray:
    out = arr.astype(np.float32, copy=True)
    if CAR_SHARE_IS_PERCENT:
        out = out / 100.0
    out = np.where(np.isfinite(out), out, 0.0)
    out = np.clip(out, 0.0, 1.0)
    return out


def _eta(seconds: float) -> str:
    if (not np.isfinite(seconds)) or seconds < 0:
        return "n/a"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"


def _score_huff(a: float, tmin: float) -> float:
    # t=0 (same-pixel reseller) is valid and should be preferred.
    if (not np.isfinite(tmin)) or (tmin < 0):
        return -np.inf
    return float(a / ((tmin + EPS) ** BETA))


def _write_float_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_FLOAT).astype(np.float32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


def _write_int_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="int32", count=1, nodata=NODATA_INT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_INT).astype(np.int32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


def _read_reseller_ids(gdf: gpd.GeoDataFrame) -> np.ndarray:
    if RESELLER_ID_COLUMN not in gdf.columns:
        raise KeyError(f"Missing column '{RESELLER_ID_COLUMN}' in reseller layer.")

    rid = pd.to_numeric(gdf[RESELLER_ID_COLUMN], errors="coerce")
    if rid.isna().any():
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' contains non-numeric values.")
    if not rid.is_unique:
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' must be unique.")
    if (rid <= 0).any():
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' must contain positive IDs.")

    return rid.astype(np.int64).to_numpy()


def _candidate_idx_adaptive(r: int, c: int, tree: cKDTree, coords: np.ndarray) -> np.ndarray:
    idx = np.array([], dtype=np.int32)

    found_primary = tree.query_ball_point([r, c], r=PRIMARY_SEARCH_RADIUS_KM, p=2)
    if len(found_primary) > 0:
        idx = np.asarray(found_primary, dtype=np.int32)

    if idx.size < MIN_CANDIDATES:
        found_max = tree.query_ball_point([r, c], r=MAX_SEARCH_RADIUS_KM, p=2)
        if len(found_max) > 0:
            idx = np.unique(np.concatenate([idx, np.asarray(found_max, dtype=np.int32)]))

    if idx.size < MIN_CANDIDATES:
        k = min(max(MIN_CANDIDATES, EXTRA_CANDIDATES_CAR), len(coords))
        _, nn = tree.query([r, c], k=k)
        idx = np.unique(np.concatenate([idx, np.atleast_1d(nn).astype(np.int32)]))

    return idx


def _winner_from_dist(dist_row: np.ndarray, cand_idx: np.ndarray, reseller_node: np.ndarray, reseller_id: np.ndarray, reseller_attr: np.ndarray):
    cand_nodes = reseller_node[cand_idx]
    t = dist_row[cand_nodes]
    a = reseller_attr[cand_idx]
    scores = np.array([_score_huff(aa, tt) for aa, tt in zip(a, t)], dtype=np.float64)
    if np.all(~np.isfinite(scores)):
        return NODATA_INT, np.nan
    j_local = int(np.nanargmax(scores))
    rid = int(reseller_id[cand_idx[j_local]])
    tmin = float(t[j_local]) if np.isfinite(t[j_local]) else np.nan
    return rid, tmin


def _run_mode_with_fallback(
    src_node: int,
    r: int,
    c: int,
    graph: csr_matrix,
    friction_min: float,
    base_idx: np.ndarray,
    r_rows: np.ndarray,
    r_cols: np.ndarray,
    reseller_node: np.ndarray,
    reseller_id: np.ndarray,
    reseller_attr: np.ndarray,
    src_component: int,
    reseller_component: np.ndarray,
    initial_limit_factor: float,
    final_limit_factor: float,
    ):
    cand = np.unique(base_idx)
    if cand.size > 0:
        cand = cand[reseller_component[cand] == src_component]

    if cand.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN, False, "no_candidate_in_component"

    lb = np.hypot(r_rows[cand] - r, r_cols[cand] - c) * CELL_SIZE_METERS * max(friction_min, EPS)
    limit = float(np.nanmax(lb) * initial_limit_factor + LIMIT_MARGIN_MIN) if lb.size > 0 else np.inf
    dist_row = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit,
    )[0]
    rid, tmin = _winner_from_dist(dist_row, cand, reseller_node, reseller_id, reseller_attr)
    if rid >= 0 and np.isfinite(tmin):
        return rid, float(tmin), False, "base_ok"

    cand_global = np.where(reseller_component == src_component)[0].astype(np.int32)
    if cand_global.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN, True, "no_global_in_component"

    lb2 = np.hypot(r_rows[cand_global] - r, r_cols[cand_global] - c) * CELL_SIZE_METERS * max(friction_min, EPS)
    limit2 = float(np.nanmax(lb2) * final_limit_factor + LIMIT_MARGIN_MIN) if lb2.size > 0 else np.inf
    dist_row2 = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit2,
    )[0]
    rid2, tmin2 = _winner_from_dist(dist_row2, cand_global, reseller_node, reseller_id, reseller_attr)
    if rid2 >= 0 and np.isfinite(tmin2):
        return rid2, float(tmin2), True, "fallback_ok"

    return NODATA_INT, UNASSIGNED_TIME_MIN, True, "fallback_unassigned"


def _bump_reason(counter: dict, key: str):
    counter[key] = int(counter.get(key, 0)) + 1


# =====================================================================
# LOAD / ALIGN INPUTS
# =====================================================================
t0 = time.time()
print("[1/8] Loading population reference raster...")
pop, ref_profile = _read_raster(POPULATION_RASTER)
transform = ref_profile["transform"]
crs = ref_profile["crs"]
height, width = pop.shape
print(f"Grid: {width} x {height}, CRS={crs}")

print("[2/8] Aligning car share and frictions to reference grid...")
car_share_raw = _align_to_reference(CAR_SHARE_RASTER, ref_profile, Resampling.nearest)
walk_friction = _align_to_reference(WALK_FRICTION_RASTER, ref_profile, Resampling.bilinear)
moto_friction = _align_to_reference(MOTO_FRICTION_RASTER, ref_profile, Resampling.bilinear)

car_share = _to_fraction(car_share_raw)
walk_share = (1.0 - car_share).astype(np.float32)

walk_friction = np.where(walk_friction > 0, walk_friction, np.nan).astype(np.float32)
moto_friction = np.where(moto_friction > 0, moto_friction, np.nan).astype(np.float32)

walk_friction_min = float(np.nanpercentile(walk_friction[np.isfinite(walk_friction)], 5))
moto_friction_min = float(np.nanpercentile(moto_friction[np.isfinite(moto_friction)], 5))

print(f"Walk friction range (min/m): {np.nanmin(walk_friction):.6f} .. {np.nanmax(walk_friction):.6f}")
print(f"Moto friction range (min/m): {np.nanmin(moto_friction):.6f} .. {np.nanmax(moto_friction):.6f}")

# =====================================================================
# LOAD RESELLERS
# =====================================================================
print("[3/8] Loading reseller points...")
resellers = gpd.read_file(RESELLER_GPKG, layer=RESELLER_LAYER)
if resellers.empty:
    raise RuntimeError("Reseller layer is empty.")
if resellers.crs != crs:
    resellers = resellers.to_crs(crs)

if ATTRACTIVENESS_COLUMN not in resellers.columns:
    raise KeyError(f"Missing column '{ATTRACTIVENESS_COLUMN}' in reseller layer.")

resellers = resellers[resellers.geometry.notna()].copy()
resellers = resellers[resellers.geometry.geom_type.isin(["Point"])].copy()
if resellers.empty:
    raise RuntimeError("No point geometries in reseller layer.")

resellers[ATTRACTIVENESS_COLUMN] = _safe_attractiveness(resellers[ATTRACTIVENESS_COLUMN])

r_rows, r_cols = rasterio.transform.rowcol(
    transform, resellers.geometry.x.values, resellers.geometry.y.values
)
r_rows = np.asarray(r_rows, dtype=np.int32)
r_cols = np.asarray(r_cols, dtype=np.int32)

inside = (r_rows >= 0) & (r_rows < height) & (r_cols >= 0) & (r_cols < width)
resellers = resellers.loc[inside].copy()
r_rows = r_rows[inside]
r_cols = r_cols[inside]

if resellers.empty:
    raise RuntimeError("No reseller is inside raster extent.")

reseller_id = _read_reseller_ids(resellers)
reseller_id = reseller_id.astype(np.int32)
print(f"Reseller id column: {RESELLER_ID_COLUMN}")

reseller_attr = resellers[ATTRACTIVENESS_COLUMN].astype(np.float64).to_numpy()

coords_rc = np.column_stack([r_rows, r_cols]).astype(np.float64)
reseller_tree = cKDTree(coords_rc)
print(f"Resellers on grid: {len(resellers)}")

# =====================================================================
# BUILD SHORTEST-PATH GRAPHS (walk and motorized)
# =====================================================================
print("[4/8] Building graph topology from valid cells...")
# Travel graph must represent traversability, not habitation.
valid = np.isfinite(walk_friction) & np.isfinite(moto_friction)
node_id = -np.ones((height, width), dtype=np.int32)
vr, vc = np.where(valid)
n_nodes = len(vr)
if n_nodes == 0:
    raise RuntimeError("No valid nodes available for graph.")
node_id[vr, vc] = np.arange(n_nodes, dtype=np.int32)
print(f"Valid graph nodes: {n_nodes:,}")

if USE_8_NEIGHBORS:
    neighbors = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
else:
    neighbors = [(-1,0),(1,0),(0,-1),(0,1)]

edge_i = []
edge_j = []
edge_cost_walk = []
edge_cost_moto = []

diag_factor = np.sqrt(2.0)

for r, c in zip(vr, vc):
    n0 = node_id[r, c]
    fw0 = walk_friction[r, c]
    fm0 = moto_friction[r, c]
    for dr, dc in neighbors:
        rr = r + dr
        cc = c + dc
        if rr < 0 or rr >= height or cc < 0 or cc >= width:
            continue
        n1 = node_id[rr, cc]
        if n1 < 0:
            continue
        fw1 = walk_friction[rr, cc]
        fm1 = moto_friction[rr, cc]
        if (not np.isfinite(fw1)) or (not np.isfinite(fm1)):
            continue

        step_m = CELL_SIZE_METERS
        if dr != 0 and dc != 0:
            step_m *= diag_factor

        cw = 0.5 * (fw0 + fw1) * step_m
        cm = 0.5 * (fm0 + fm1) * step_m
        if cw <= 0 or cm <= 0:
            continue

        edge_i.append(n0)
        edge_j.append(n1)
        edge_cost_walk.append(float(cw))
        edge_cost_moto.append(float(cm))

graph_walk = csr_matrix((edge_cost_walk, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
graph_moto = csr_matrix((edge_cost_moto, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
print(f"Directed edges: {len(edge_i):,}")

n_comp_walk, comp_walk = connected_components(csgraph=graph_walk, directed=False, return_labels=True)
n_comp_moto, comp_moto = connected_components(csgraph=graph_moto, directed=False, return_labels=True)
print(f"Connected components (walk/moto): {n_comp_walk:,} / {n_comp_moto:,}")

# map resellers to graph nodes
reseller_node = node_id[r_rows, r_cols]
valid_res = reseller_node >= 0
if not np.any(valid_res):
    raise RuntimeError("No reseller mapped to valid graph node.")
reseller_node = reseller_node[valid_res]
reseller_id = reseller_id[valid_res]
reseller_attr = reseller_attr[valid_res]
r_rows = r_rows[valid_res]
r_cols = r_cols[valid_res]
coords_rc = np.column_stack([r_rows, r_cols]).astype(np.float64)
reseller_tree = cKDTree(coords_rc)
reseller_comp_walk = comp_walk[reseller_node]
reseller_comp_moto = comp_moto[reseller_node]

# =====================================================================
# INHABITED PIXELS
# =====================================================================
print("[5/8] Preparing inhabited pixel list...")
inhabited = np.isfinite(pop) & (pop > MIN_POP_PER_PIXEL) & valid
pix_rows, pix_cols = np.where(inhabited)
n_pix = len(pix_rows)
if n_pix == 0:
    raise RuntimeError("No inhabited pixels found.")
if MAX_PIXELS_DEBUG is not None:
    n_use = min(MAX_PIXELS_DEBUG, n_pix)
    pix_rows = pix_rows[:n_use]
    pix_cols = pix_cols[:n_use]
    n_pix = n_use
    print(f"DEBUG mode active: {n_pix} pixels")
print(f"Inhabited pixels to process: {n_pix:,}")

best_id_walk = np.full((height, width), NODATA_INT, dtype=np.int32)
best_time_walk = np.full((height, width), np.nan, dtype=np.float32)
best_id_car = np.full((height, width), NODATA_INT, dtype=np.int32)
best_time_car = np.full((height, width), np.nan, dtype=np.float32)

profile = {
    "enabled": bool(ENABLE_PROFILING),
    "start_unix": float(time.time()),
    "config": {
        "progress_every": int(PROGRESS_EVERY),
        "profile_block_pixels": int(PROFILE_BLOCK_PIXELS),
        "save_support_rasters": bool(SAVE_SUPPORT_RASTERS),
        "max_pixels_debug": None if MAX_PIXELS_DEBUG is None else int(MAX_PIXELS_DEBUG),
        "primary_search_radius_km": int(PRIMARY_SEARCH_RADIUS_KM),
        "max_search_radius_km": int(MAX_SEARCH_RADIUS_KM),
    },
    "counts": {
        "pixels_total": int(n_pix),
        "walk_fallback_used": 0,
        "car_fallback_used": 0,
        "car_copied_from_walk": 0,
    },
    "timings_sec": {
        "total": 0.0,
        "pixel_total": 0.0,
        "walk_mode": 0.0,
        "car_mode": 0.0,
    },
    "fallback_reason_walk": {},
    "fallback_reason_car": {},
    "block_stats": [],
}

block_t0 = time.time()
block_k0 = 0

# =====================================================================
# MAIN LOOP (shortest-path per pixel source)
# =====================================================================
print("[6/8] Running Huff assignment with shortest-path travel times...")
loop_t0 = time.time()

for k, (r, c) in enumerate(zip(pix_rows, pix_cols), start=1):
    pixel_t0 = time.time()

    src_node = node_id[r, c]
    if src_node < 0:
        continue

    cand_walk = _candidate_idx_adaptive(int(r), int(c), reseller_tree, coords_rc)
    src_comp_walk = int(comp_walk[int(src_node)])

    tw0 = time.time()
    rid_w, t_w, used_fb_w, reason_w = _run_mode_with_fallback(
        src_node=int(src_node),
        r=int(r),
        c=int(c),
        graph=graph_walk,
        friction_min=walk_friction_min,
        base_idx=cand_walk,
        r_rows=r_rows,
        r_cols=r_cols,
        reseller_node=reseller_node,
        reseller_id=reseller_id,
        reseller_attr=reseller_attr,
        src_component=src_comp_walk,
        reseller_component=reseller_comp_walk,
        initial_limit_factor=INITIAL_LIMIT_FACTOR_WALK,
        final_limit_factor=FINAL_LIMIT_FACTOR_WALK,
    )
    profile["timings_sec"]["walk_mode"] += float(time.time() - tw0)
    if used_fb_w:
        profile["counts"]["walk_fallback_used"] += 1
    _bump_reason(profile["fallback_reason_walk"], reason_w)

    best_id_walk[r, c] = rid_w
    best_time_walk[r, c] = t_w

    cs = float(car_share[r, c])
    expected_car_users = cs * float(pop[r, c])
    if expected_car_users < MIN_EXPECTED_CAR_USERS_PER_PIXEL:
        best_id_car[r, c] = rid_w
        best_time_car[r, c] = t_w
        profile["counts"]["car_copied_from_walk"] += 1
    else:
        k_extra = min(EXTRA_CANDIDATES_CAR, len(coords_rc))
        _, nn = reseller_tree.query([int(r), int(c)], k=k_extra)
        nn = np.atleast_1d(nn).astype(np.int32)
        cand_car = np.unique(np.concatenate([cand_walk, nn]))
        src_comp_moto = int(comp_moto[int(src_node)])

        tc0 = time.time()
        rid_c, t_c, used_fb_c, reason_c = _run_mode_with_fallback(
            src_node=int(src_node),
            r=int(r),
            c=int(c),
            graph=graph_moto,
            friction_min=moto_friction_min,
            base_idx=cand_car,
            r_rows=r_rows,
            r_cols=r_cols,
            reseller_node=reseller_node,
            reseller_id=reseller_id,
            reseller_attr=reseller_attr,
            src_component=src_comp_moto,
            reseller_component=reseller_comp_moto,
            initial_limit_factor=INITIAL_LIMIT_FACTOR_CAR,
            final_limit_factor=FINAL_LIMIT_FACTOR_CAR,
        )
        profile["timings_sec"]["car_mode"] += float(time.time() - tc0)
        if used_fb_c:
            profile["counts"]["car_fallback_used"] += 1
        _bump_reason(profile["fallback_reason_car"], reason_c)

        best_id_car[r, c] = rid_c
        best_time_car[r, c] = t_c

    profile["timings_sec"]["pixel_total"] += float(time.time() - pixel_t0)

    if ENABLE_PROFILING and ((k % PROFILE_BLOCK_PIXELS == 0) or (k == n_pix)):
        blk_elapsed = float(time.time() - block_t0)
        blk_pix = int(k - block_k0)
        profile["block_stats"].append(
            {
                "k_end": int(k),
                "pixels": blk_pix,
                "elapsed_sec": blk_elapsed,
                "pix_per_sec": float(blk_pix / max(blk_elapsed, 1e-9)),
            }
        )
        block_t0 = time.time()
        block_k0 = k

    if (k % PROGRESS_EVERY == 0) or (k == n_pix):
        elapsed = time.time() - loop_t0
        speed = k / max(elapsed, 1e-9)
        rem = (n_pix - k) / max(speed, 1e-9)
        total_est = elapsed + rem
        if total_est > 300:
            print(f"  {k:,}/{n_pix:,} ({100.0*k/n_pix:.1f}%) | {speed:.1f} pix/s | ETA {_eta(rem)}")
        else:
            print(f"  {k:,}/{n_pix:,} ({100.0*k/n_pix:.1f}%) | {speed:.1f} pix/s")

# =====================================================================
# OUTPUTS
# =====================================================================
print("[7/8] Writing GPKG output...")
out_rows = pix_rows.astype(np.int32)
out_cols = pix_cols.astype(np.int32)
xs = transform.c + (out_cols + 0.5) * transform.a
ys = transform.f + (out_rows + 0.5) * transform.e

df = pd.DataFrame({
    "row": out_rows,
    "col": out_cols,
    "x": xs.astype(np.float64),
    "y": ys.astype(np.float64),
    "car_share": car_share[out_rows, out_cols].astype(np.float32),
    "walk_share": walk_share[out_rows, out_cols].astype(np.float32),
    "best_reseller_id_walk": best_id_walk[out_rows, out_cols].astype(np.int32),
    "best_time_walk_min": best_time_walk[out_rows, out_cols].astype(np.float32),
    "best_reseller_id_car": best_id_car[out_rows, out_cols].astype(np.int32),
    "best_time_car_min": best_time_car[out_rows, out_cols].astype(np.float32),
})

gdf_out = gpd.GeoDataFrame(
    df,
    geometry=[Point(x, y) for x, y in zip(df["x"].to_numpy(), df["y"].to_numpy())],
    crs=crs,
)
gdf_out.to_file(OUTPUT_GPKG, layer=OUTPUT_GPKG_LAYER, driver="GPKG")
print(f"Saved: {OUTPUT_GPKG} | layer={OUTPUT_GPKG_LAYER}")

lookup = pd.DataFrame({
    "reseller_id": reseller_id,
    "attractiveness": reseller_attr,
    "row": r_rows,
    "col": r_cols,
}).drop_duplicates(subset=["reseller_id"])
lookup.to_csv(OUTPUT_LOOKUP_CSV, index=False)
print(f"Saved lookup: {OUTPUT_LOOKUP_CSV}")

print("[8/8] Writing support rasters...")
if SAVE_SUPPORT_RASTERS:
    _write_int_raster(OUTPUT_RASTER_BEST_ID_WALK, best_id_walk.astype(np.float32), ref_profile)
    _write_float_raster(OUTPUT_RASTER_BEST_TIME_WALK, best_time_walk, ref_profile)
    _write_int_raster(OUTPUT_RASTER_BEST_ID_CAR, best_id_car.astype(np.float32), ref_profile)
    _write_float_raster(OUTPUT_RASTER_BEST_TIME_CAR, best_time_car, ref_profile)
    print("Support rasters saved.")
else:
    print("Support rasters skipped.")

valid_walk = (best_id_walk[inhabited] >= 0) & np.isfinite(best_time_walk[inhabited]) & (best_time_walk[inhabited] < UNASSIGNED_TIME_MIN)
valid_car = (best_id_car[inhabited] >= 0) & np.isfinite(best_time_car[inhabited]) & (best_time_car[inhabited] < UNASSIGNED_TIME_MIN)

profile["timings_sec"]["total"] = float(time.time() - t0)
profile["counts"]["walk_assigned"] = int(valid_walk.sum())
profile["counts"]["car_assigned"] = int(valid_car.sum())
profile["counts"]["walk_unassigned"] = int(n_pix - valid_walk.sum())
profile["counts"]["car_unassigned"] = int(n_pix - valid_car.sum())

print("\n=== SUMMARY ===")
print(f"Pixels processed: {n_pix:,}")
print(f"Walk assigned pixels: {int(valid_walk.sum()):,} ({100.0 * valid_walk.mean():.2f}%)")
print(f"Car assigned pixels : {int(valid_car.sum()):,} ({100.0 * valid_car.mean():.2f}%)")
if valid_walk.any():
    w = best_time_walk[inhabited][valid_walk]
    print(f"Walk time min/median/max (min): {np.nanmin(w):.2f} / {np.nanmedian(w):.2f} / {np.nanmax(w):.2f}")
if valid_car.any():
    c = best_time_car[inhabited][valid_car]
    print(f"Car  time min/median/max (min): {np.nanmin(c):.2f} / {np.nanmedian(c):.2f} / {np.nanmax(c):.2f}")

if ENABLE_PROFILING:
    with open(PROFILE_LOG_JSON, "w", encoding="utf-8") as f:
        json.dump(profile, f, indent=2)

    print("\n=== PROFILING (light) ===")
    print(f"Walk fallback used: {profile['counts']['walk_fallback_used']:,} / {n_pix:,}")
    print(f"Car fallback used : {profile['counts']['car_fallback_used']:,} / {n_pix:,}")
    print(f"Car copied from walk: {profile['counts']['car_copied_from_walk']:,}")
    print(f"Timing log JSON: {PROFILE_LOG_JSON}")

print(f"\nDone in {_eta(time.time() - t0)}")

[1/8] Loading population reference raster...
Grid: 1337 x 1085, CRS=EPSG:3857
[2/8] Aligning car share and frictions to reference grid...
Walk friction range (min/m): 0.012000 .. 0.162533
Moto friction range (min/m): 0.000500 .. 0.159416
[3/8] Loading reseller points...
Reseller id column: id_res&fil
Resellers on grid: 2792
[4/8] Building graph topology from valid cells...
Valid graph nodes: 941,966
Directed edges: 3,758,344
Connected components (walk/moto): 15 / 15
[5/8] Preparing inhabited pixel list...
Inhabited pixels to process: 563,482
[6/8] Running Huff assignment with shortest-path travel times...
  2,000/563,482 (0.4%) | 14.8 pix/s | ETA 10h 31m 16s
  4,000/563,482 (0.7%) | 15.3 pix/s | ETA 10h 11m 4s
  6,000/563,482 (1.1%) | 15.4 pix/s | ETA 10h 2m 4s
  8,000/563,482 (1.4%) | 16.4 pix/s | ETA 9h 24m 32s
  10,000/563,482 (1.8%) | 17.0 pix/s | ETA 9h 3m 0s
  12,000/563,482 (2.1%) | 17.4 pix/s | ETA 8h 47m 8s
  14,000/563,482 (2.5%) | 17.4 pix/s | ETA 8h 47m 44s
  16,000/563,482

In [2]:
"""
Optional post-process aligned with id_res&fil workflow.

This cell only refreshes a compact reseller lookup table from the source GPKG:
- reseller_id (id_res&fil)
- attractiveness
- row, col on Population.tif grid

No fid/place_id decoding and no extra place-id rasters.
"""

import pandas as pd
import geopandas as gpd
import rasterio

DATA_DIR = "dataset_big"
RESELLER_GPKG = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"
RESELLER_LAYER = "resell_and_filling"
POPULATION_RASTER = f"{DATA_DIR}/Population.tif"
LOOKUP_CSV = f"{DATA_DIR}/huff_reseller_lookup.csv"

RESELLER_ID_COLUMN = "id_res&fil"
ATTRACTIVENESS_COLUMN = "attractiveness"

print("[post] Loading reseller layer...")
resellers = gpd.read_file(RESELLER_GPKG, layer=RESELLER_LAYER)
if resellers.empty:
    raise RuntimeError("Reseller layer is empty.")

if RESELLER_ID_COLUMN not in resellers.columns:
    raise KeyError(f"Missing column '{RESELLER_ID_COLUMN}' in reseller layer.")
if ATTRACTIVENESS_COLUMN not in resellers.columns:
    raise KeyError(f"Missing column '{ATTRACTIVENESS_COLUMN}' in reseller layer.")

resellers = resellers[resellers.geometry.notna()].copy()
resellers = resellers[resellers.geometry.geom_type.isin(["Point"])].copy()
if resellers.empty:
    raise RuntimeError("No point geometries in reseller layer.")

rid = pd.to_numeric(resellers[RESELLER_ID_COLUMN], errors="coerce")
if rid.isna().any() or (rid <= 0).any() or (not rid.is_unique):
    raise ValueError(f"Column '{RESELLER_ID_COLUMN}' must be unique positive numeric IDs.")

with rasterio.open(POPULATION_RASTER) as src:
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width

if resellers.crs != crs:
    resellers = resellers.to_crs(crs)

rows, cols = rasterio.transform.rowcol(transform, resellers.geometry.x.values, resellers.geometry.y.values)
rows = pd.to_numeric(pd.Series(rows), errors="coerce").fillna(-1).astype(int).to_numpy()
cols = pd.to_numeric(pd.Series(cols), errors="coerce").fillna(-1).astype(int).to_numpy()
inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)

lookup = pd.DataFrame(
    {
        "reseller_id": rid.astype("int32").to_numpy()[inside],
        "attractiveness": pd.to_numeric(resellers[ATTRACTIVENESS_COLUMN], errors="coerce").fillna(0).to_numpy()[inside],
        "row": rows[inside],
        "col": cols[inside],
    }
).drop_duplicates(subset=["reseller_id"])

lookup.to_csv(LOOKUP_CSV, index=False)
print(f"[post] Lookup CSV updated: {LOOKUP_CSV}")
print(f"[post] Rows written: {len(lookup):,}")

[post] Loading reseller layer...
[post] Lookup CSV updated: dataset_big/huff_reseller_lookup.csv
[post] Rows written: 2,792
